In [1]:
# from pathlib import Path
# import gc
# import json
# import time
# from typing import Dict, List, Tuple

# import numpy as np
# import pandas as pd
# import soundfile as sf
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from tqdm.auto import tqdm

# from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
# from comet import download_model, load_from_checkpoint
# from numcodecs import Quantize, Zstd


# # ============================================================
# # CELL 1: Imports and global config
# # ============================================================

# PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
# OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "experiment_results"
# EXPERIMENT_NAME = "custom_selected_linear_structured_pruning_codec_eval_only_en_zh"
# EXP_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
# EXP_DIR.mkdir(parents=True, exist_ok=True)

# MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
# PROMPT_TEXT = "Translate this English speech into Chinese."
# MAX_NEW_TOKENS = 64
# COMET_BATCH_SIZE = 16

# EVAL_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_eval_en_zh.csv"

# PRUNE_RATIO = 0.20
# PRUNE_SCORING = "magnitude"
# APPLY_CODEC_AFTER_PRUNING = True
# CODEC_DIGITS = 2
# CODEC_ZSTD_LEVEL = 3

# print("project root exists:", PROJECT_ROOT.exists())
# print("cuda available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("gpu:", torch.cuda.get_device_name(0))


# # ============================================================
# # CELL 2: Load eval fold, processor, COMET
# # ============================================================

# eval_df = pd.read_csv(EVAL_MANIFEST).copy()
# eval_df["audio_path"] = eval_df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))

# processor = AutoProcessor.from_pretrained(MODEL_ID)

# comet_path = download_model("Unbabel/wmt22-comet-da")
# comet_model = load_from_checkpoint(comet_path)

# print("eval rows:", len(eval_df))
# print(eval_df.head(2))


# # ============================================================
# # CELL 3: Codec and compressed linear layer
# # ============================================================

# class CodecPipeline:
#     def __init__(self, filters, compressor=None, encoded_dtype="f2"):
#         self.filters = filters
#         self.compressor = compressor
#         self.encoded_dtype = np.dtype(encoded_dtype)

#     def encode(self, arr):
#         x = np.asarray(arr, dtype=np.float32)

#         for f in self.filters:
#             x = f.encode(x)

#         x = np.asarray(x, dtype=self.encoded_dtype)
#         filtered_shape = x.shape
#         filtered_bytes = x.tobytes(order="C")

#         if self.compressor is not None:
#             filtered_bytes = self.compressor.encode(filtered_bytes)

#         return {
#             "payload": filtered_bytes,
#             "filtered_shape": filtered_shape,
#             "filtered_dtype": self.encoded_dtype.str,
#         }

#     def decode(self, encoded_obj, original_shape, original_dtype=np.float32):
#         payload = encoded_obj["payload"]
#         filtered_shape = tuple(encoded_obj["filtered_shape"])
#         filtered_dtype = np.dtype(encoded_obj["filtered_dtype"])

#         if self.compressor is not None:
#             payload = self.compressor.decode(payload)

#         x = np.frombuffer(payload, dtype=filtered_dtype).copy().reshape(filtered_shape)

#         for f in reversed(self.filters):
#             x = f.decode(x)

#         x = np.asarray(x, dtype=original_dtype).reshape(original_shape)
#         return x


# class CompressedLinear(nn.Module):
#     def __init__(self, linear_layer, pipeline, description):
#         super().__init__()

#         weight = linear_layer.weight.detach().cpu().numpy().astype(np.float32)

#         self.weight_shape = weight.shape
#         self.in_features = linear_layer.in_features
#         self.out_features = linear_layer.out_features
#         self.pipeline = pipeline
#         self.description = description

#         self.encoded_weight = self.pipeline.encode(weight)

#         if linear_layer.bias is not None:
#             self.bias = nn.Parameter(linear_layer.bias.detach().clone(), requires_grad=False)
#         else:
#             self.bias = None

#         self._cached_weight = None
#         self._cached_device = None
#         self._cached_dtype = None

#     def _get_cached_weight(self, device, dtype):
#         if (
#             self._cached_weight is None
#             or self._cached_device != device
#             or self._cached_dtype != dtype
#         ):
#             decoded = self.pipeline.decode(
#                 self.encoded_weight,
#                 original_shape=self.weight_shape,
#                 original_dtype=np.float32,
#             )
#             self._cached_weight = torch.tensor(decoded, device=device, dtype=dtype)
#             self._cached_device = device
#             self._cached_dtype = dtype

#         return self._cached_weight

#     def forward(self, x):
#         weight = self._get_cached_weight(x.device, x.dtype)
#         return F.linear(x, weight, self.bias)


# # ============================================================
# # CELL 4: Model navigation helpers
# # ============================================================

# def get_module_by_name(model, module_name):
#     module = model
#     for part in module_name.split("."):
#         module = getattr(module, part)
#     return module


# def set_module_by_name(model, module_name, new_module):
#     parts = module_name.split(".")
#     parent = model
#     for part in parts[:-1]:
#         parent = getattr(parent, part)
#     setattr(parent, parts[-1], new_module)


# def get_linear_layer_names(model):
#     names = []
#     for name, module in model.named_modules():
#         if isinstance(module, nn.Linear):
#             names.append(name)
#     return names


# def should_select_layer(layer_name):
#     selected_patterns = [
#         "mlp",
#         "feed_forward",
#         "up_proj",
#         "down_proj",
#         "gate_proj",
#     ]
#     return any(pattern in layer_name for pattern in selected_patterns)


# def get_selected_linear_layer_names(model):
#     return [name for name in get_linear_layer_names(model) if should_select_layer(name)]


# def estimate_linear_storage_bytes(layer):
#     if isinstance(layer, nn.Linear):
#         total = layer.weight.detach().cpu().numpy().nbytes
#         if layer.bias is not None:
#             total += layer.bias.detach().cpu().numpy().nbytes
#         return total
#     if isinstance(layer, CompressedLinear):
#         total = len(layer.encoded_weight["payload"])
#         if layer.bias is not None:
#             total += layer.bias.detach().cpu().numpy().nbytes
#         return total
#     return 0


# def cleanup_cuda():
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()


# # ============================================================
# # CELL 5: Inference helpers
# # ============================================================

# def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS):
#     audio, sr = sf.read(str(audio_path))

#     if audio.ndim > 1:
#         audio = audio.mean(axis=1)

#     conversation = [
#         {
#             "role": "user",
#             "content": [
#                 {"type": "audio", "audio_url": str(audio_path)},
#                 {"type": "text", "text": prompt_text},
#             ],
#         }
#     ]

#     text = processor.apply_chat_template(
#         conversation,
#         add_generation_prompt=True,
#         tokenize=False,
#     )

#     inputs = processor(
#         text=text,
#         audios=[audio],
#         sampling_rate=sr,
#         return_tensors="pt",
#     )

#     model_device = next(model.parameters()).device
#     inputs = {k: v.to(model_device) if hasattr(v, "to") else v for k, v in inputs.items()}

#     with torch.no_grad():
#         generated_ids = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=False,
#             num_beams=1,
#             use_cache=True,
#         )

#     generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

#     pred = processor.batch_decode(
#         generated_ids,
#         skip_special_tokens=True,
#         clean_up_tokenization_spaces=False,
#     )[0].strip()

#     return pred


# def run_inference(model, df_input, run_name, max_new_tokens=MAX_NEW_TOKENS):
#     preds = []
#     start = time.time()

#     for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
#         pred = generate_translation(
#             model,
#             processor,
#             Path(row["audio_path"]),
#             max_new_tokens=max_new_tokens,
#         )
#         preds.append(pred)

#     elapsed = time.time() - start

#     out_df = df_input.copy()
#     out_df["prediction"] = preds
#     return out_df, elapsed


# def compute_comet(pred_df, comet_model, batch_size=COMET_BATCH_SIZE):
#     comet_data = [
#         {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
#         for _, r in pred_df.iterrows()
#     ]

#     comet_out = comet_model.predict(comet_data, batch_size=batch_size, gpus=1)
#     comet_score = comet_out.system_score if hasattr(comet_out, "system_score") else comet_out[1]
#     return comet_score


# # ============================================================
# # CELL 6: Structured pruning helpers
# # ============================================================

# def make_codec_pipeline():
#     return CodecPipeline(
#         filters=[Quantize(digits=CODEC_DIGITS, dtype="f4", astype="f2")],
#         compressor=Zstd(level=CODEC_ZSTD_LEVEL),
#         encoded_dtype="f2",
#     )


# def make_new_linear_from_old(old_layer: nn.Linear, keep_out_idxs=None, keep_in_idxs=None) -> nn.Linear:
#     device = old_layer.weight.device
#     dtype = old_layer.weight.dtype

#     weight = old_layer.weight.detach()
#     bias = old_layer.bias.detach() if old_layer.bias is not None else None

#     if keep_out_idxs is None:
#         keep_out_idxs = torch.arange(weight.shape[0], device=weight.device)
#     if keep_in_idxs is None:
#         keep_in_idxs = torch.arange(weight.shape[1], device=weight.device)

#     keep_out_idxs = keep_out_idxs.to(weight.device)
#     keep_in_idxs = keep_in_idxs.to(weight.device)

#     new_weight = weight.index_select(0, keep_out_idxs).index_select(1, keep_in_idxs).contiguous()

#     new_bias = None
#     if bias is not None:
#         new_bias = bias.index_select(0, keep_out_idxs).contiguous()

#     new_layer = nn.Linear(
#         in_features=new_weight.shape[1],
#         out_features=new_weight.shape[0],
#         bias=new_bias is not None,
#     )
#     new_layer = new_layer.to(device=device, dtype=dtype)

#     with torch.no_grad():
#         new_layer.weight.copy_(new_weight.to(device=device, dtype=dtype))
#         if new_bias is not None:
#             new_layer.bias.copy_(new_bias.to(device=device, dtype=dtype))

#     return new_layer


# def find_mlp_triplets(model) -> List[Dict[str, str]]:
#     linear_names = set(get_linear_layer_names(model))
#     triplets = []

#     for layer_name in sorted(linear_names):
#         if not layer_name.endswith(".gate_proj"):
#             continue
#         prefix = layer_name[: -len(".gate_proj")]
#         up_name = prefix + ".up_proj"
#         down_name = prefix + ".down_proj"
#         if up_name in linear_names and down_name in linear_names:
#             triplets.append(
#                 {
#                     "block_prefix": prefix,
#                     "gate_proj": layer_name,
#                     "up_proj": up_name,
#                     "down_proj": down_name,
#                 }
#             )
#     return triplets


# def score_triplet_units(gate_layer: nn.Linear, up_layer: nn.Linear, down_layer: nn.Linear, method: str = PRUNE_SCORING) -> torch.Tensor:
#     gate_w = gate_layer.weight.detach().float()
#     up_w = up_layer.weight.detach().float()
#     down_w = down_layer.weight.detach().float()

#     if method == "magnitude":
#         gate_score = gate_w.abs().mean(dim=1)
#         up_score = up_w.abs().mean(dim=1)
#         down_score = down_w.abs().mean(dim=0)
#         return gate_score + up_score + down_score

#     raise ValueError(f"Unsupported PRUNE_SCORING={method}")


# def compute_keep_indices(scores: torch.Tensor, prune_ratio: float) -> Tuple[torch.Tensor, int]:
#     width = int(scores.numel())
#     n_prune = int(np.floor(width * prune_ratio))
#     n_prune = min(max(n_prune, 0), max(width - 1, 0))
#     n_keep = width - n_prune
#     keep_idxs = torch.topk(scores, k=n_keep, largest=True).indices.sort().values
#     return keep_idxs, n_prune


# def apply_selected_layer_structured_pruning(model, prune_ratio: float = PRUNE_RATIO):
#     triplets = find_mlp_triplets(model)
#     pruning_rows = []

#     print("mlp triplets found for structured pruning:", len(triplets))

#     for triplet in tqdm(triplets, desc="apply_selected_layer_structured_pruning"):
#         gate_name = triplet["gate_proj"]
#         up_name = triplet["up_proj"]
#         down_name = triplet["down_proj"]

#         gate_layer = get_module_by_name(model, gate_name)
#         up_layer = get_module_by_name(model, up_name)
#         down_layer = get_module_by_name(model, down_name)

#         if not all(isinstance(x, nn.Linear) for x in [gate_layer, up_layer, down_layer]):
#             raise TypeError(f"Expected nn.Linear triplet at {triplet['block_prefix']}")

#         scores = score_triplet_units(gate_layer, up_layer, down_layer, method=PRUNE_SCORING)
#         keep_idxs, n_pruned = compute_keep_indices(scores, prune_ratio=prune_ratio)

#         new_gate = make_new_linear_from_old(gate_layer, keep_out_idxs=keep_idxs)
#         new_up = make_new_linear_from_old(up_layer, keep_out_idxs=keep_idxs)
#         new_down = make_new_linear_from_old(down_layer, keep_in_idxs=keep_idxs)

#         set_module_by_name(model, gate_name, new_gate)
#         set_module_by_name(model, up_name, new_up)
#         set_module_by_name(model, down_name, new_down)

#         pruning_rows.append(
#             {
#                 "block_prefix": triplet["block_prefix"],
#                 "gate_proj": gate_name,
#                 "up_proj": up_name,
#                 "down_proj": down_name,
#                 "original_intermediate_size": int(scores.numel()),
#                 "kept_intermediate_size": int(keep_idxs.numel()),
#                 "pruned_intermediate_size": int(n_pruned),
#                 "prune_ratio": float(prune_ratio),
#                 "scoring_method": PRUNE_SCORING,
#             }
#         )

#     return pd.DataFrame(pruning_rows)


# def apply_codec_to_selected_layers(model):
#     selected_layers = get_selected_linear_layer_names(model)
#     codec_rows = []
#     pipeline = make_codec_pipeline()
#     desc = f"Selected-layer codec after structured pruning: Quantize(digits={CODEC_DIGITS}, astype='f2') + Zstd"

#     for layer_name in tqdm(selected_layers, desc="apply_codec_to_selected_layers"):
#         layer = get_module_by_name(model, layer_name)
#         if not isinstance(layer, nn.Linear):
#             continue
#         compressed_layer = CompressedLinear(layer, pipeline, desc)
#         compressed_layer = compressed_layer.to(next(model.parameters()).device)
#         set_module_by_name(model, layer_name, compressed_layer)
#         codec_rows.append(
#             {
#                 "layer_name": layer_name,
#                 "compression_method": desc,
#             }
#         )

#     return pd.DataFrame(codec_rows)


# # ============================================================
# # CELL 7: Load baseline model on GPU
# # ============================================================

# baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.float16,
#     device_map="auto",
# )
# baseline_model.eval()


# # ============================================================
# # CELL 8: Record original linear sizes before pruning/compression
# # ============================================================

# original_linear_bytes = {}
# for layer_name in get_linear_layer_names(baseline_model):
#     layer = get_module_by_name(baseline_model, layer_name)
#     if isinstance(layer, nn.Linear):
#         original_linear_bytes[layer_name] = estimate_linear_storage_bytes(layer)

# original_selected_layers = get_selected_linear_layer_names(baseline_model)
# selected_layers_path = EXP_DIR / "selected_layers_before_pruning.json"
# with open(selected_layers_path, "w", encoding="utf-8") as f:
#     json.dump(original_selected_layers, f, indent=2)
# print("selected layers saved to:", selected_layers_path)


# # ============================================================
# # CELL 9: Apply selected-layer structured pruning in place
# # ============================================================

# pruning_df = apply_selected_layer_structured_pruning(baseline_model, prune_ratio=PRUNE_RATIO)
# pruning_path = EXP_DIR / "selected_layer_structured_pruning_policy.csv"
# pruning_df.to_csv(pruning_path, index=False)
# print("structured pruning policy saved to:", pruning_path)
# print(pruning_df.head())

# codec_df = pd.DataFrame(columns=["layer_name", "compression_method"])
# if APPLY_CODEC_AFTER_PRUNING:
#     codec_df = apply_codec_to_selected_layers(baseline_model)
#     codec_path = EXP_DIR / "selected_layer_codec_after_pruning_policy.csv"
#     codec_df.to_csv(codec_path, index=False)
#     print("codec-after-pruning policy saved to:", codec_path)
#     print(codec_df.head())


# # ============================================================
# # CELL 10: Storage report
# # ============================================================

# storage_rows = []
# total_original = 0
# total_current = 0

# for layer_name in get_linear_layer_names(baseline_model):
#     layer = get_module_by_name(baseline_model, layer_name)
#     orig_bytes = original_linear_bytes.get(layer_name, np.nan)
#     curr_bytes = estimate_linear_storage_bytes(layer)

#     if not np.isnan(orig_bytes):
#         total_original += orig_bytes
#     total_current += curr_bytes

#     storage_rows.append(
#         {
#             "layer_name": layer_name,
#             "is_selected_layer": should_select_layer(layer_name),
#             "original_bytes": orig_bytes,
#             "current_bytes": curr_bytes,
#             "compression_ratio": (orig_bytes / curr_bytes) if (not np.isnan(orig_bytes) and curr_bytes > 0) else np.nan,
#             "method": getattr(layer, "description", "structured_pruned_or_original"),
#             "module_type": type(layer).__name__,
#         }
#     )

# storage_report_df = pd.DataFrame(storage_rows)
# storage_report_path = EXP_DIR / "selected_linear_structured_pruning_codec_storage_report.csv"
# storage_report_df.to_csv(storage_report_path, index=False)

# storage_totals = {
#     "original_total_bytes": total_original,
#     "current_total_bytes": total_current,
#     "total_ratio": total_original / total_current if total_current > 0 else np.nan,
# }

# print("storage report saved to:", storage_report_path)
# print(storage_totals)


# # ============================================================
# # CELL 11: Run eval on pruned/compressed model
# # ============================================================

# experiment_model = baseline_model
# experiment_model.eval()

# pred_df, eval_time = run_inference(
#     experiment_model,
#     eval_df,
#     run_name="selected_linear_structured_pruning_codec_eval",
#     max_new_tokens=MAX_NEW_TOKENS,
# )

# preds_path = EXP_DIR / "selected_linear_structured_pruning_codec_eval_preds.csv"
# pred_df.to_csv(preds_path, index=False, encoding="utf-8")

# comet_score = compute_comet(pred_df, comet_model)

# print("predictions saved to:", preds_path)
# print("eval seconds:", round(eval_time, 2))
# print("eval COMET:", comet_score)


# # ============================================================
# # CELL 12: Save summary CSV
# # ============================================================

# timestamp_utc = pd.Timestamp.utcnow().isoformat()

# summary_df = pd.DataFrame(
#     [
#         {
#             "timestamp_utc": timestamp_utc,
#             "experiment_name": EXPERIMENT_NAME,
#             "run": "selected_linear_structured_pruning_codec_eval",
#             "split": "eval",
#             "rows": len(pred_df),
#             "seconds": eval_time,
#             "seconds_per_item": eval_time / len(pred_df),
#             "comet": comet_score,
#             "model_id": MODEL_ID,
#             "prompt_text": PROMPT_TEXT,
#             "max_new_tokens": MAX_NEW_TOKENS,
#             "comet_batch_size": COMET_BATCH_SIZE,
#             "predictions_csv": str(preds_path),
#             "method": "selected_layer_structured_pruning" + ("_plus_codec" if APPLY_CODEC_AFTER_PRUNING else ""),
#             "prune_ratio": PRUNE_RATIO,
#             "prune_scoring": PRUNE_SCORING,
#             "apply_codec_after_pruning": APPLY_CODEC_AFTER_PRUNING,
#             "codec_digits": CODEC_DIGITS if APPLY_CODEC_AFTER_PRUNING else np.nan,
#             "codec_zstd_level": CODEC_ZSTD_LEVEL if APPLY_CODEC_AFTER_PRUNING else np.nan,
#             "original_total_bytes": storage_totals["original_total_bytes"],
#             "current_total_bytes": storage_totals["current_total_bytes"],
#             "compression_ratio": storage_totals["total_ratio"],
#             "selected_layers_json": str(selected_layers_path),
#             "pruning_policy_csv": str(pruning_path),
#             "storage_report_csv": str(storage_report_path),
#         }
#     ]
# )

# summary_path = EXP_DIR / "selected_linear_structured_pruning_codec_eval_summary.csv"
# summary_df.to_csv(summary_path, index=False)

# print("summary saved to:", summary_path)
# print(summary_df)


# # ============================================================
# # CELL 13: Cleanup
# # ============================================================

# del experiment_model
# cleanup_cuda()


In [3]:
from pathlib import Path
import gc
import json
import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
from comet import download_model, load_from_checkpoint
from numcodecs import Quantize, Zstd


# ============================================================
# CONFIG
# ============================================================

PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "experiment_results"
EXPERIMENT_NAME = "custom_selected_linear_structured_pruning_codec_eval_only_en_zh"
EXP_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
PROMPT_TEXT = "Translate this English speech into Chinese."
MAX_NEW_TOKENS = 64
COMET_BATCH_SIZE = 16

EVAL_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_eval_en_zh.csv"

PRUNE_RATIO = 0.20
PRUNE_SCORING = "magnitude"
APPLY_CODEC_AFTER_PRUNING = True
CODEC_DIGITS = 2
CODEC_ZSTD_LEVEL = 3


# ============================================================
# LOAD EVAL DATA, PROCESSOR, COMET
# ============================================================

eval_df = pd.read_csv(EVAL_MANIFEST).copy()
eval_df["audio_path"] = eval_df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))

processor = AutoProcessor.from_pretrained(MODEL_ID)

comet_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_path)

print("project root exists:", PROJECT_ROOT.exists())
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("eval rows:", len(eval_df))


# ============================================================
# CODEC AND COMPRESSED LINEAR
# ============================================================

class CodecPipeline:
    def __init__(self, filters, compressor=None, encoded_dtype="f2"):
        self.filters = filters
        self.compressor = compressor
        self.encoded_dtype = np.dtype(encoded_dtype)

    def encode(self, arr):
        x = np.asarray(arr, dtype=np.float32)
        for f in self.filters:
            x = f.encode(x)
        x = np.asarray(x, dtype=self.encoded_dtype)
        filtered_shape = x.shape
        filtered_bytes = x.tobytes(order="C")
        if self.compressor is not None:
            filtered_bytes = self.compressor.encode(filtered_bytes)
        return {
            "payload": filtered_bytes,
            "filtered_shape": filtered_shape,
            "filtered_dtype": self.encoded_dtype.str,
        }

    def decode(self, encoded_obj, original_shape, original_dtype=np.float32):
        payload = encoded_obj["payload"]
        filtered_shape = tuple(encoded_obj["filtered_shape"])
        filtered_dtype = np.dtype(encoded_obj["filtered_dtype"])

        if self.compressor is not None:
            payload = self.compressor.decode(payload)

        x = np.frombuffer(payload, dtype=filtered_dtype).copy().reshape(filtered_shape)

        for f in reversed(self.filters):
            x = f.decode(x)

        x = np.asarray(x, dtype=original_dtype).reshape(original_shape)
        return x


class CompressedLinear(nn.Module):
    def __init__(self, linear_layer, pipeline, description):
        super().__init__()

        weight = linear_layer.weight.detach().cpu().numpy().astype(np.float32)

        self.weight_shape = weight.shape
        self.in_features = linear_layer.in_features
        self.out_features = linear_layer.out_features
        self.pipeline = pipeline
        self.description = description

        self.encoded_weight = self.pipeline.encode(weight)

        if linear_layer.bias is not None:
            self.bias = nn.Parameter(linear_layer.bias.detach().clone(), requires_grad=False)
        else:
            self.bias = None

        self._cached_weight = None
        self._cached_device = None
        self._cached_dtype = None

    def _get_cached_weight(self, device, dtype):
        if (
            self._cached_weight is None
            or self._cached_device != device
            or self._cached_dtype != dtype
        ):
            decoded = self.pipeline.decode(
                self.encoded_weight,
                original_shape=self.weight_shape,
                original_dtype=np.float32,
            )
            self._cached_weight = torch.tensor(decoded, device=device, dtype=dtype)
            self._cached_device = device
            self._cached_dtype = dtype
        return self._cached_weight

    def forward(self, x):
        weight = self._get_cached_weight(x.device, x.dtype)
        return F.linear(x, weight, self.bias)


# ============================================================
# MODEL NAVIGATION HELPERS
# ============================================================

def get_module_by_name(model, module_name):
    module = model
    for part in module_name.split("."):
        module = getattr(module, part)
    return module


def set_module_by_name(model, module_name, new_module):
    parts = module_name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


def get_linear_layer_names(model):
    names = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            names.append(name)
    return names


def should_select_layer(layer_name):
    selected_patterns = [
        "mlp",
        "feed_forward",
        "up_proj",
        "down_proj",
        "gate_proj",
    ]
    return any(pattern in layer_name for pattern in selected_patterns)


def get_selected_linear_layer_names(model):
    return [name for name in get_linear_layer_names(model) if should_select_layer(name)]


def estimate_linear_storage_bytes(layer):
    if isinstance(layer, nn.Linear):
        total = layer.weight.detach().cpu().numpy().nbytes
        if layer.bias is not None:
            total += layer.bias.detach().cpu().numpy().nbytes
        return int(total)
    if isinstance(layer, CompressedLinear):
        total = len(layer.encoded_weight["payload"])
        if layer.bias is not None:
            total += layer.bias.detach().cpu().numpy().nbytes
        return int(total)
    return 0


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# INFERENCE HELPERS
# ============================================================

def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS):
    audio, sr = sf.read(str(audio_path))

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": str(audio_path)},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        text=text,
        audios=[audio],
        sampling_rate=sr,
        return_tensors="pt",
    )

    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) if hasattr(v, "to") else v for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

    pred = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return pred


def run_inference(model, df_input, run_name, max_new_tokens=MAX_NEW_TOKENS):
    preds = []
    start = time.time()

    for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
        pred = generate_translation(
            model,
            processor,
            Path(row["audio_path"]),
            max_new_tokens=max_new_tokens,
        )
        preds.append(pred)

    elapsed = time.time() - start

    out_df = df_input.copy()
    out_df["prediction"] = preds
    return out_df, elapsed


def compute_comet(pred_df, comet_model, batch_size=COMET_BATCH_SIZE):
    comet_data = [
        {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
        for _, r in pred_df.iterrows()
    ]

    comet_out = comet_model.predict(comet_data, batch_size=batch_size, gpus=1)
    comet_score = comet_out.system_score if hasattr(comet_out, "system_score") else comet_out[1]
    return comet_score


# ============================================================
# STRUCTURED PRUNING HELPERS
# ============================================================

def make_codec_pipeline():
    return CodecPipeline(
        filters=[Quantize(digits=CODEC_DIGITS, dtype="f4", astype="f2")],
        compressor=Zstd(level=CODEC_ZSTD_LEVEL),
        encoded_dtype="f2",
    )


def make_new_linear_from_old(old_layer: nn.Linear, keep_out_idxs=None, keep_in_idxs=None) -> nn.Linear:
    device = old_layer.weight.device
    dtype = old_layer.weight.dtype

    weight = old_layer.weight.detach()
    bias = old_layer.bias.detach() if old_layer.bias is not None else None

    if keep_out_idxs is None:
        keep_out_idxs = torch.arange(weight.shape[0], device=weight.device)
    if keep_in_idxs is None:
        keep_in_idxs = torch.arange(weight.shape[1], device=weight.device)

    keep_out_idxs = keep_out_idxs.to(weight.device)
    keep_in_idxs = keep_in_idxs.to(weight.device)

    new_weight = weight.index_select(0, keep_out_idxs).index_select(1, keep_in_idxs).contiguous()

    new_bias = None
    if bias is not None:
        new_bias = bias.index_select(0, keep_out_idxs).contiguous()

    new_layer = nn.Linear(
        in_features=new_weight.shape[1],
        out_features=new_weight.shape[0],
        bias=new_bias is not None,
    )
    new_layer = new_layer.to(device=device, dtype=dtype)

    with torch.no_grad():
        new_layer.weight.copy_(new_weight.to(device=device, dtype=dtype))
        if new_bias is not None:
            new_layer.bias.copy_(new_bias.to(device=device, dtype=dtype))

    return new_layer


def find_mlp_triplets(model) -> List[Dict[str, str]]:
    linear_names = set(get_linear_layer_names(model))
    triplets = []

    for layer_name in sorted(linear_names):
        if not layer_name.endswith(".gate_proj"):
            continue
        prefix = layer_name[: -len(".gate_proj")]
        up_name = prefix + ".up_proj"
        down_name = prefix + ".down_proj"
        if up_name in linear_names and down_name in linear_names:
            triplets.append(
                {
                    "block_prefix": prefix,
                    "gate_proj": layer_name,
                    "up_proj": up_name,
                    "down_proj": down_name,
                }
            )
    return triplets


def score_triplet_units(gate_layer: nn.Linear, up_layer: nn.Linear, down_layer: nn.Linear, method: str = PRUNE_SCORING) -> torch.Tensor:
    gate_w = gate_layer.weight.detach().float()
    up_w = up_layer.weight.detach().float()
    down_w = down_layer.weight.detach().float()

    if method == "magnitude":
        gate_score = gate_w.abs().mean(dim=1)
        up_score = up_w.abs().mean(dim=1)
        down_score = down_w.abs().mean(dim=0)
        return gate_score + up_score + down_score

    raise ValueError(f"Unsupported PRUNE_SCORING={method}")


def compute_keep_indices(scores: torch.Tensor, prune_ratio: float) -> Tuple[torch.Tensor, int]:
    width = int(scores.numel())
    n_prune = int(np.floor(width * prune_ratio))
    n_prune = min(max(n_prune, 0), max(width - 1, 0))
    n_keep = width - n_prune
    keep_idxs = torch.topk(scores, k=n_keep, largest=True).indices.sort().values
    return keep_idxs, n_prune


def apply_selected_layer_structured_pruning(model, prune_ratio: float = PRUNE_RATIO):
    triplets = find_mlp_triplets(model)
    pruning_rows = []

    print("mlp triplets found for structured pruning:", len(triplets))

    for triplet in tqdm(triplets, desc="apply_selected_layer_structured_pruning"):
        gate_name = triplet["gate_proj"]
        up_name = triplet["up_proj"]
        down_name = triplet["down_proj"]

        gate_layer = get_module_by_name(model, gate_name)
        up_layer = get_module_by_name(model, up_name)
        down_layer = get_module_by_name(model, down_name)

        if not all(isinstance(x, nn.Linear) for x in [gate_layer, up_layer, down_layer]):
            raise TypeError(f"Expected nn.Linear triplet at {triplet['block_prefix']}")

        scores = score_triplet_units(gate_layer, up_layer, down_layer, method=PRUNE_SCORING)
        keep_idxs, n_pruned = compute_keep_indices(scores, prune_ratio=prune_ratio)

        new_gate = make_new_linear_from_old(gate_layer, keep_out_idxs=keep_idxs)
        new_up = make_new_linear_from_old(up_layer, keep_out_idxs=keep_idxs)
        new_down = make_new_linear_from_old(down_layer, keep_in_idxs=keep_idxs)

        set_module_by_name(model, gate_name, new_gate)
        set_module_by_name(model, up_name, new_up)
        set_module_by_name(model, down_name, new_down)

        pruning_rows.append(
            {
                "block_prefix": triplet["block_prefix"],
                "gate_proj": gate_name,
                "up_proj": up_name,
                "down_proj": down_name,
                "original_intermediate_size": int(scores.numel()),
                "kept_intermediate_size": int(keep_idxs.numel()),
                "pruned_intermediate_size": int(n_pruned),
                "prune_ratio": float(prune_ratio),
                "scoring_method": PRUNE_SCORING,
            }
        )

    return pd.DataFrame(pruning_rows)


def apply_codec_to_selected_layers(model, selected_layer_names=None):
    if selected_layer_names is None:
        selected_layer_names = get_selected_linear_layer_names(model)

    codec_rows = []
    pipeline = make_codec_pipeline()
    desc = (
        f"Selected-layer codec after structured pruning: Quantize(digits={CODEC_DIGITS}, astype='f2') + Zstd"
    )

    for layer_name in tqdm(selected_layer_names, desc="apply_codec_to_selected_layers"):
        layer = get_module_by_name(model, layer_name)
        if not isinstance(layer, nn.Linear):
            continue
        compressed_layer = CompressedLinear(layer, pipeline, desc)
        compressed_layer = compressed_layer.to(next(model.parameters()).device)
        set_module_by_name(model, layer_name, compressed_layer)
        codec_rows.append(
            {
                "layer_name": layer_name,
                "compression_method": desc,
            }
        )

    return pd.DataFrame(codec_rows)


# ============================================================
# MAIN EXPERIMENT
# ============================================================

def main():
    baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    baseline_model.eval()

    # Record original bytes for all original linear layers.
    original_linear_bytes = {}
    original_linear_types = {}
    original_linear_names = get_linear_layer_names(baseline_model)
    for layer_name in original_linear_names:
        layer = get_module_by_name(baseline_model, layer_name)
        if isinstance(layer, nn.Linear):
            original_linear_bytes[layer_name] = estimate_linear_storage_bytes(layer)
            original_linear_types[layer_name] = type(layer).__name__

    # Preserve selected-layer names BEFORE replacing modules.
    original_selected_layers = get_selected_linear_layer_names(baseline_model)
    selected_layers_path = EXP_DIR / "selected_layers_before_pruning.json"
    with open(selected_layers_path, "w", encoding="utf-8") as f:
        json.dump(original_selected_layers, f, indent=2)
    print("selected layers saved to:", selected_layers_path)

    # Apply structured pruning.
    pruning_df = apply_selected_layer_structured_pruning(baseline_model, prune_ratio=PRUNE_RATIO)
    pruning_path = EXP_DIR / "selected_layer_structured_pruning_policy.csv"
    pruning_df.to_csv(pruning_path, index=False)
    print("structured pruning policy saved to:", pruning_path)
    print(pruning_df.head())

    # Apply codec after pruning over the preserved selected-layer names.
    codec_df = pd.DataFrame(columns=["layer_name", "compression_method"])
    if APPLY_CODEC_AFTER_PRUNING:
        codec_df = apply_codec_to_selected_layers(baseline_model, selected_layer_names=original_selected_layers)
        codec_path = EXP_DIR / "selected_layer_codec_after_pruning_policy.csv"
        codec_df.to_csv(codec_path, index=False)
        print("codec-after-pruning policy saved to:", codec_path)
        print(codec_df.head())

    # Storage report over the ORIGINAL linear-layer universe, not only current nn.Linear modules.
    storage_rows = []
    total_original = 0
    total_current = 0

    for layer_name in original_linear_names:
        orig_bytes = original_linear_bytes.get(layer_name, np.nan)

        try:
            layer = get_module_by_name(baseline_model, layer_name)
            curr_bytes = estimate_linear_storage_bytes(layer)
            method_name = getattr(layer, "description", "structured_pruned_or_original")
            module_type = type(layer).__name__
        except AttributeError:
            layer = None
            curr_bytes = np.nan
            method_name = "missing_after_replacement"
            module_type = "missing"

        if not np.isnan(orig_bytes):
            total_original += orig_bytes
        if not np.isnan(curr_bytes):
            total_current += curr_bytes

        storage_rows.append(
            {
                "layer_name": layer_name,
                "is_selected_layer": layer_name in set(original_selected_layers),
                "original_linear_bytes_est": orig_bytes,
                "current_linear_bytes_est": curr_bytes,
                "compression_ratio_est": (orig_bytes / curr_bytes)
                if (not np.isnan(orig_bytes) and not np.isnan(curr_bytes) and curr_bytes > 0)
                else np.nan,
                "method": method_name,
                "module_type": module_type,
                "original_module_type": original_linear_types.get(layer_name, "unknown"),
            }
        )

    storage_report_df = pd.DataFrame(storage_rows)
    storage_report_path = EXP_DIR / "selected_linear_structured_pruning_codec_storage_report.csv"
    storage_report_df.to_csv(storage_report_path, index=False)

    storage_totals = {
        "size_scope": "estimated_original_linear_layers",
        "original_total_linear_bytes_est": int(total_original),
        "current_total_linear_bytes_est": int(total_current),
        "compression_ratio_est": (total_original / total_current) if total_current > 0 else np.nan,
    }

    print("storage report saved to:", storage_report_path)
    print(storage_totals)

    # Eval.
    experiment_model = baseline_model
    experiment_model.eval()

    pred_df, eval_time = run_inference(
        experiment_model,
        eval_df,
        run_name="selected_linear_structured_pruning_codec_eval",
        max_new_tokens=MAX_NEW_TOKENS,
    )

    preds_path = EXP_DIR / "selected_linear_structured_pruning_codec_eval_preds.csv"
    pred_df.to_csv(preds_path, index=False, encoding="utf-8")

    comet_score = compute_comet(pred_df, comet_model)

    print("predictions saved to:", preds_path)
    print("eval seconds:", round(eval_time, 2))
    print("eval COMET:", comet_score)

    # Summary.
    timestamp_utc = pd.Timestamp.utcnow().isoformat()

    summary_df = pd.DataFrame(
        [
            {
                "timestamp_utc": timestamp_utc,
                "experiment_name": EXPERIMENT_NAME,
                "run": "selected_linear_structured_pruning_codec_eval",
                "split": "eval",
                "rows": len(pred_df),
                "seconds": eval_time,
                "seconds_per_item": eval_time / len(pred_df),
                "comet": comet_score,
                "model_id": MODEL_ID,
                "prompt_text": PROMPT_TEXT,
                "max_new_tokens": MAX_NEW_TOKENS,
                "comet_batch_size": COMET_BATCH_SIZE,
                "predictions_csv": str(preds_path),
                "method": "selected_layer_structured_pruning_plus_codec"
                if APPLY_CODEC_AFTER_PRUNING else "selected_layer_structured_pruning",
                "prune_ratio": PRUNE_RATIO,
                "prune_scoring": PRUNE_SCORING,
                "apply_codec_after_pruning": APPLY_CODEC_AFTER_PRUNING,
                "codec_digits": CODEC_DIGITS if APPLY_CODEC_AFTER_PRUNING else np.nan,
                "codec_zstd_level": CODEC_ZSTD_LEVEL if APPLY_CODEC_AFTER_PRUNING else np.nan,
                "size_scope": storage_totals["size_scope"],
                "original_total_linear_bytes_est": storage_totals["original_total_linear_bytes_est"],
                "current_total_linear_bytes_est": storage_totals["current_total_linear_bytes_est"],
                "compression_ratio_est": storage_totals["compression_ratio_est"],
                "selected_layers_json": str(selected_layers_path),
                "pruning_policy_csv": str(pruning_path),
                "storage_report_csv": str(storage_report_path),
            }
        ]
    )

    summary_path = EXP_DIR / "selected_linear_structured_pruning_codec_eval_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print("summary saved to:", summary_path)
    print(summary_df)

    del experiment_model
    cleanup_cuda()


if __name__ == "__main__":
    main()


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/lightning_fabric/utilities/cloud_io.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torc

project root exists: True
cuda available: True
gpu: NVIDIA RTX A6000
eval rows: 416


We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

...

apply_codec_to_selected_layers:   0%|          | 0/96 [00:00<?, ?it/s]

codec-after-pruning policy saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_selected_linear_structured_pruning_codec_eval_only_en_zh/selected_layer_codec_after_pruning_policy.csv
                                    layer_name  \
0  language_model.model.layers.0.mlp.gate_proj   
1    language_model.model.layers.0.mlp.up_proj   
2  language_model.model.layers.0.mlp.down_proj   
3  language_model.model.layers.1.mlp.gate_proj   
4    language_model.model.layers.1.mlp.up_proj   

                                  compression_method  
0  Selected-layer codec after structured pruning:...  
1  Selected-layer codec after structured pruning:...  
2  Selected-layer codec after structured pruning:...  
3  Selected-layer codec after structured pruning:...  
4  Selected-layer codec after structured pruning:...  
storage report saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_selected_linear_structured_pruning_codec

selected_linear_structured_pruning_codec_eval:   0%|          | 0/416 [00:00<?, ?it/s]

/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
We

predictions saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_selected_linear_structured_pruning_codec_eval_only_en_zh/selected_linear_structured_pruning_codec_eval_preds.csv
eval seconds: 356.14
eval COMET: 0.5871149682296584
summary saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_selected_linear_structured_pruning_codec_eval_only_en_zh/selected_linear_structured_pruning_codec_eval_summary.csv
                      timestamp_utc  \
0  2026-03-20T14:53:04.104166+00:00   

                                     experiment_name  \
0  custom_selected_linear_structured_pruning_code...   

                                             run split  rows     seconds  \
0  selected_linear_structured_pruning_codec_eval  eval   416  356.140175   

   seconds_per_item     comet                      model_id  \
0          0.856106  0.587115  Qwen/Qwen2-Audio-7B-Instruct   

                                   prompt_te